# WSmart+ Route — Data Analysis

End-to-end analysis of the two dataset types used in the project:

| Part | Format | Location | Purpose |
|------|--------|----------|---------|
| **A — TensorDict** | `.td` | `data/datasets/<problem>/` | Training & validation instances |
| **B — NPZ Simulator** | `.npz` | `data/wsr_simulator/datasets/` | Realistic multi-day simulation scenarios |

Run cells top-to-bottom inside each part independently.

In [ ]:
from notebook_setup import setup_google_colab, setup_home_directory

NOTEBOOK_NAME = 'data_analysis'
PROJECT_ROOT = setup_home_directory(NOTEBOOK_NAME)
IN_COLAB, gdrive, gfiles = setup_google_colab(NOTEBOOK_NAME)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch
from scipy.stats import gaussian_kde

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({
    'figure.dpi': 130, 'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
})
CMAP = 'viridis'
COLORS = ['steelblue', 'darkorange', 'mediumseagreen', 'tomato', 'mediumpurple']
print(f'PyTorch {torch.__version__}  |  device: {DEVICE}')

---
## Part A — TensorDict Training Datasets

Files: `data/datasets/<problem>/<problem><N>_<dist>_time<T>_seed<seed>.td`  
Loaded as a [`TensorDict`](https://pytorch.org/tensordict/) — a typed batch of tensors sharing a batch dimension.

### A-1 · Configuration

In [ ]:
# ── Edit these to select a different dataset ─────────────────────────────────
PROBLEM_NAME      = 'vrpp'      # problem subdirectory
GRAPH_SIZE        = 100         # number of client nodes (20 / 50 / 100 / 170)
DATA_DISTRIBUTION = 'gamma3'    # 'gamma1' | 'gamma3' | 'emp'
TIME_LIMIT        = 100         # 100 → training set  |  7 → validation set
TD_SEED           = 42

DATASET_PATH = os.path.join(
    PROJECT_ROOT, 'data', 'datasets', PROBLEM_NAME,
    f'{PROBLEM_NAME}{GRAPH_SIZE}_{DATA_DISTRIBUTION}_time{TIME_LIMIT}_seed{TD_SEED}.td',
)
print('Path :', DATASET_PATH)
print('Exists:', os.path.exists(DATASET_PATH))

### A-2 · Load & Inspect

In [ ]:
td = torch.load(DATASET_PATH, map_location='cpu', weights_only=False)
B = td.batch_size[0]
locs  = td['locs'].numpy()   # (B, N, 2)
depot = td['depot'].numpy()  # (B, 2)
N     = locs.shape[1]

# Classify keys into node-level scalars vs instance-level constants
NODE_SCALARS = {}
INST_SCALARS = {}
for k in td.keys():
    if k in ('depot', 'locs'):
        continue
    arr = td[k].numpy()
    if arr.ndim == 2 and arr.shape == (B, N):
        NODE_SCALARS[k] = arr
    elif arr.ndim == 3 and arr.shape[0] == B and arr.shape[2] == N:
        NODE_SCALARS[k] = arr.mean(axis=1)
        print(f'  Note: {k!r} shaped {arr.shape} (time-series) -> averaged to {NODE_SCALARS[k].shape}')
    else:
        INST_SCALARS[k] = arr

# Summary table
rows = []
for k in td.keys():
    t = td[k].float()
    rows.append({'Key': k, 'Shape': tuple(t.shape), 'Dtype': str(td[k].dtype).replace('torch.',''),
                 'Min': round(float(t.min()), 4), 'Max': round(float(t.max()), 4),
                 'Mean': round(float(t.mean()), 4), 'Std': round(float(t.std()), 4),
                 'MB': round(t.element_size() * t.numel() / 1e6, 2)})
display(pd.DataFrame(rows).set_index('Key'))
print(f'\nBatch: {B}  |  Nodes: {N}')
print(f'Node scalars  : {list(NODE_SCALARS)}')
print(f'Inst constants: {list(INST_SCALARS)}')

### A-3 · Spatial Analysis

In [ ]:
rng  = np.random.default_rng(SEED)
picks = rng.choice(B, size=4, replace=False)
color_key = next(iter(NODE_SCALARS), None)

def _plot_instance(node_xy, depot_xy, c=None, clabel='', title='', ax=None):
    standalone = ax is None
    if standalone:
        _, ax = plt.subplots(figsize=(5, 5))
    if c is not None:
        sc = ax.scatter(node_xy[:,0], node_xy[:,1], c=c, cmap=CMAP, s=60,
                        edgecolors='white', linewidth=0.4, zorder=2)
        plt.colorbar(sc, ax=ax, shrink=0.7, label=clabel, pad=0.02)
    else:
        ax.scatter(node_xy[:,0], node_xy[:,1], s=60, alpha=0.7, edgecolors='white',
                   linewidth=0.4, zorder=2)
    ax.scatter(*depot_xy, c='red', s=250, marker='*', edgecolors='black', linewidth=0.8,
               zorder=3, label='Depot')
    ax.set_title(title, fontsize=10); ax.set_aspect('equal')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax.legend(fontsize=8, loc='upper right', framealpha=0.8)
    if standalone:
        plt.tight_layout(); plt.show()
    return ax

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, idx in zip(axes.ravel(), picks):
    c_vals = NODE_SCALARS[color_key][int(idx)] if color_key else None
    _plot_instance(locs[int(idx)], depot[int(idx)], c=c_vals, clabel=color_key or '',
                   title=f'Instance {int(idx)}  ({N} nodes)', ax=ax)
plt.suptitle(f'Sample Instances — {os.path.basename(DATASET_PATH)}', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# Coordinate density + depot distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
all_x, all_y = locs[:,:,0].ravel(), locs[:,:,1].ravel()

hb = axes[0].hexbin(all_x, all_y, gridsize=30, cmap='Blues')
fig.colorbar(hb, ax=axes[0], label='Count')
axes[0].set_title('Node Location Density (hex-bin)')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

axes[1].hist(all_x, bins=40, alpha=0.7, color='steelblue', label='x')
axes[1].hist(all_y, bins=40, alpha=0.7, color='coral', label='y')
axes[1].set_title('Marginal Coordinate Distributions')
axes[1].set_xlabel('Value'); axes[1].legend()

axes[2].scatter(depot[:,0], depot[:,1], s=12, alpha=0.4, color='red')
axes[2].set_title(f'Depot Positions (n={B})'); axes[2].set_aspect('equal')
axes[2].set_xlim(0,1); axes[2].set_ylim(0,1)

plt.tight_layout(); plt.show()

# Nearest-neighbour + depot distances
def _nn_dist(locs, k=1):
    diff = locs[:,: ,None,:] - locs[:,None,:,:]
    dist = np.linalg.norm(diff, axis=-1)
    np.fill_diagonal(dist.reshape(B, N*N).T, np.inf)
    return np.sort(dist, axis=-1)[:,:,:k].mean(axis=(-1,-2))

nn   = _nn_dist(locs)
ddep = np.linalg.norm(locs - depot[:,None,:], axis=-1).ravel()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(nn, bins=30, color='teal', alpha=0.8, edgecolor='white')
axes[0].axvline(nn.mean(), color='red', linestyle='--', label=f'Mean={nn.mean():.4f}')
axes[0].set_title('Mean 1-NN Distance per Instance'); axes[0].legend()

axes[1].hist(ddep, bins=40, color='darkorange', alpha=0.8, edgecolor='white')
axes[1].axvline(ddep.mean(), color='red', linestyle='--', label=f'Mean={ddep.mean():.4f}')
axes[1].set_title('Node-to-Depot Distance Distribution'); axes[1].legend()
plt.tight_layout(); plt.show()

### A-4 · Waste Distributions

In [ ]:
if not NODE_SCALARS:
    print('No node-level scalars found in this dataset — skipping waste analysis.')
else:
    for key, arr in NODE_SCALARS.items():
        flat = arr.ravel()
        x_r  = np.linspace(float(flat.min()), float(flat.max()), 400)
        kde  = gaussian_kde(flat, bw_method='scott')(x_r)

        fig, axes = plt.subplots(1, 3, figsize=(18, 4))

        # Histogram + KDE
        axes[0].hist(flat, bins=60, density=True, color='steelblue', alpha=0.35,
                     edgecolor='none', label='Histogram')
        axes[0].plot(x_r, kde, color='steelblue', lw=2.5, label='KDE')
        axes[0].axvline(flat.mean(),      color='red',   ls='--', lw=1.5,
                        label=f'Mean={flat.mean():.3f}')
        axes[0].axvline(np.median(flat),  color='black', ls=':',  lw=1.5,
                        label=f'Median={np.median(flat):.3f}')
        axes[0].set_title(f'`{key}` — PDF')
        axes[0].set_xlabel('Value'); axes[0].set_ylabel('Density'); axes[0].legend(fontsize=8)

        # Empirical CDF
        s = np.sort(flat)
        axes[1].plot(s, np.linspace(0, 1, len(s)), color='steelblue', lw=2)
        axes[1].set_title(f'`{key}` — Empirical CDF')
        axes[1].set_xlabel('Value'); axes[1].set_ylabel('P(X ≤ x)')

        # Per-bin mean ± std
        bm, bs = arr.mean(axis=0), arr.std(axis=0)
        x_idx  = np.arange(N)
        axes[2].bar(x_idx, bm, color='steelblue', alpha=0.7, label='Mean')
        axes[2].fill_between(x_idx, bm-bs, bm+bs, alpha=0.25, color='steelblue', label='±1 std')
        axes[2].set_title(f'`{key}` — Per-Bin Mean ± Std')
        axes[2].set_xlabel('Bin index'); axes[2].legend(fontsize=8)
        step = max(1, N//10)
        axes[2].set_xticks(x_idx[::step])

        plt.suptitle(f'{key} — {os.path.basename(DATASET_PATH)}', fontsize=12, y=1.02)
        plt.tight_layout(); plt.show()

        # Per-bin box plot (max 60 bins shown)
        step2 = max(1, N // 60)
        bin_idx = list(range(0, N, step2))
        fig2, ax = plt.subplots(figsize=(max(10, len(bin_idx)*0.45), 5))
        ax.boxplot([arr[:, j] for j in bin_idx], patch_artist=True,
                   boxprops=dict(facecolor='lightsteelblue', alpha=0.85),
                   medianprops=dict(color='red', linewidth=2),
                   flierprops=dict(marker='.', markersize=1.5, alpha=0.2))
        ax.set_xticks(range(1, len(bin_idx)+1))
        ax.set_xticklabels(bin_idx, rotation=90 if len(bin_idx)>20 else 0, fontsize=7)
        ax.set_xlabel('Bin index'); ax.set_ylabel(key)
        ax.set_title(f'`{key}` — Box & Whisker per Bin  ({B} instances)')
        plt.tight_layout(); plt.show()

### A-5 · Multi-Dataset Comparison

In [ ]:
# ── Compare across problem sizes ─────────────────────────────────────────────
GRAPH_SIZES     = [20, 50, 100, 170]
COMPARE_DIST    = DATA_DISTRIBUTION   # keep distribution fixed
COMPARE_TIME    = TIME_LIMIT

size_tds = {}
for gs in GRAPH_SIZES:
    p = os.path.join(PROJECT_ROOT, 'data', 'datasets', PROBLEM_NAME,
                     f'{PROBLEM_NAME}{gs}_{COMPARE_DIST}_time{COMPARE_TIME}_seed{TD_SEED}.td')
    if os.path.exists(p):
        ds = torch.load(p, map_location='cpu', weights_only=False)
        size_tds[gs] = ds
        print(f'  N={gs}: batch={ds.batch_size[0]}  keys={list(ds.keys())}')
    else:
        print(f'  N={gs}: not found')

# NN distance comparison
if len(size_tds) > 1:
    fig, ax = plt.subplots(figsize=(9, 4))
    for (gs, ds), col in zip(size_tds.items(), COLORS):
        l = ds['locs'].numpy(); b = l.shape[0]
        diff = l[:,: ,None,:] - l[:,None,:,:]
        d    = np.linalg.norm(diff, axis=-1)
        np.fill_diagonal(d.reshape(b, gs*gs).T, np.inf)
        nn = np.sort(d, axis=-1)[:,:,0].mean(axis=-1)
        ax.hist(nn, bins=25, alpha=0.5, color=col, density=True, label=f'N={gs}')
    ax.set_title('Mean 1-NN Distance — Across Problem Sizes')
    ax.set_xlabel('Distance'); ax.set_ylabel('Density'); ax.legend()
    plt.tight_layout(); plt.show()

In [ ]:
# ── Compare waste distributions across data distributions ──────────────────────
DISTRIBUTIONS  = ['gamma1', 'gamma3', 'emp']
DIST_COLORS    = ['steelblue', 'darkorange', 'mediumseagreen']
DIST_LABELS    = {'gamma1': 'Gamma-1', 'gamma3': 'Gamma-3', 'emp': 'Empirical'}
COMPARE_SIZE   = GRAPH_SIZE

dist_wastes = {}
for dist in DISTRIBUTIONS:
    p = os.path.join(PROJECT_ROOT, 'data', 'datasets', PROBLEM_NAME,
                     f'{PROBLEM_NAME}{COMPARE_SIZE}_{dist}_time{COMPARE_TIME}_seed{TD_SEED}.td')
    if not os.path.exists(p):
        print(f'  {dist}: not found'); continue
    ds  = torch.load(p, map_location='cpu', weights_only=False)
    if 'waste' not in ds.keys():
        print(f'  {dist}: no waste key'); continue
    arr = ds['waste'].numpy()
    dist_wastes[dist] = arr.mean(axis=1) if arr.ndim == 3 else arr
    print(f'  {dist}: {dist_wastes[dist].shape}  mean={dist_wastes[dist].mean():.4f}')

if len(dist_wastes) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    for (dist, arr), col in zip(dist_wastes.items(), DIST_COLORS):
        lbl  = DIST_LABELS.get(dist, dist)
        flat = arr.ravel()
        x_r  = np.linspace(float(flat.min()), float(flat.max()), 400)
        kde  = gaussian_kde(flat, bw_method='scott')(x_r)
        axes[0].plot(x_r, kde, color=col, lw=2, label=lbl)
        axes[0].fill_between(x_r, kde, alpha=0.12, color=col)
        sub = np.sort(np.random.choice(flat, min(50000, len(flat)), replace=False))
        axes[1].plot(sub, np.linspace(0,1,len(sub)), color=col, lw=2, label=lbl)
        axes[2].plot(np.arange(arr.shape[1]), arr.mean(axis=0), color=col, lw=2, label=lbl)
        axes[2].fill_between(np.arange(arr.shape[1]),
                             arr.mean(axis=0)-arr.std(axis=0),
                             arr.mean(axis=0)+arr.std(axis=0), alpha=0.15, color=col)

    for ax, t, xl, yl in zip(axes,
        [f'KDE  (N={COMPARE_SIZE})', f'Empirical CDF  (N={COMPARE_SIZE})',
         f'Per-Bin Mean ± Std  (N={COMPARE_SIZE})'],
        ['Waste level','Waste level','Bin index'],
        ['Density','P(X ≤ x)','Mean waste level']):
        ax.set_title(t); ax.set_xlabel(xl); ax.set_ylabel(yl); ax.legend(fontsize=9)

    plt.suptitle('Waste Distribution Comparison — Across Distributions', fontsize=12, y=1.02)
    plt.tight_layout(); plt.show()

### A-6 · DataLoader Integration

In [ ]:
from torch.utils.data import DataLoader
from logic.src.data.datasets import TensorDictDataset
from logic.src.utils.data.td_utils import tensordict_collate_fn
import time

BATCH_SIZE_DL = 32
dataset = TensorDictDataset(td)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE_DL, shuffle=True,
                     collate_fn=tensordict_collate_fn)

print(f'Dataset size      : {len(dataset)}')
print(f'Batch size        : {BATCH_SIZE_DL}')
print(f'Batches per epoch : {len(loader)}')

batch = next(iter(loader))
print('\nMini-batch shapes:')
for k in batch.keys():
    t = batch[k]
    print(f'  {k:<14} {tuple(t.shape)}  {t.dtype}')

t0 = time.perf_counter()
n_inst = sum(b.batch_size[0] for b in loader)
elapsed = time.perf_counter() - t0
print(f'\nOne epoch: {n_inst} instances in {elapsed:.2f}s  ({n_inst/elapsed:.0f} inst/s)')

---
## Part B — NPZ Simulator Datasets

Files: `data/wsr_simulator/datasets/<area><N>_<dist>_wsr<days>_N<samples>_seed<seed>.npz`

These datasets capture realistic multi-day waste fill scenarios for specific geographic areas.
Coordinates are in **geographic space** (latitude / longitude), not normalised [0, 1].

| Array | Shape | Description |
|-------|-------|-------------|
| `depot` | `(S, 2)` | Depot lat/lon per sample |
| `locs` | `(S, N, 2)` | Bin lat/lon per sample |
| `node_ids` | `(S, N)` | Real bin IDs |
| `waste` | `(S, T, N)` | True fill levels (% of capacity) per day per bin |
| `noisy_waste` | `(S, T, N)` | Fill levels with sensor noise |
| `max_waste` | `(S,)` | Bin capacity |

### B-1 · Configuration

In [ ]:
# ── Edit these to select a different NPZ dataset ─────────────────────────────
NPZ_AREA         = 'riomaior'   # area prefix in filename
NPZ_N_BINS       = 100          # number of bins (20 | 50 | 100 | 170)
NPZ_DISTRIBUTION = 'gamma3'     # 'gamma3' | 'emp'
NPZ_DAYS         = 30           # simulation length in days
NPZ_N_SAMPLES    = 1            # number of scenario samples (N)
NPZ_SEED         = 42

NPZ_PATH = os.path.join(
    PROJECT_ROOT, 'data', 'wsr_simulator', 'datasets',
    f'{NPZ_AREA}{NPZ_N_BINS}_{NPZ_DISTRIBUTION}_wsr{NPZ_DAYS}_N{NPZ_N_SAMPLES}_seed{NPZ_SEED}.npz',
)
print('Path  :', NPZ_PATH)
print('Exists:', os.path.exists(NPZ_PATH))

### B-2 · Load & Inspect

In [ ]:
npz = np.load(NPZ_PATH, allow_pickle=True)
print('Arrays in file:')
rows_npz = []
for k in npz.files:
    arr = npz[k]
    rows_npz.append({'Key': k, 'Shape': tuple(arr.shape), 'Dtype': str(arr.dtype),
                     'Min': round(float(arr.min()), 4), 'Max': round(float(arr.max()), 4),
                     'Mean': round(float(arr.mean()), 4), 'Std': round(float(arr.std()), 4),
                     'MB': round(arr.nbytes / 1e6, 2)})
display(pd.DataFrame(rows_npz).set_index('Key'))

# Convenience aliases
S = npz['locs'].shape[0]          # samples
N_B = npz['locs'].shape[1]        # bins
T_D = npz['waste'].shape[1]       # days

npz_depot  = npz['depot']         # (S, 2)  lat/lon
npz_locs   = npz['locs']          # (S, N, 2) lat/lon
npz_ids    = npz['node_ids']      # (S, N)
npz_waste  = npz['waste']         # (S, T, N)
npz_noisy  = npz['noisy_waste']   # (S, T, N)
npz_cap    = npz['max_waste']     # (S,)

print(f'\nSamples: {S}  |  Bins: {N_B}  |  Days: {T_D}')
print(f'Bin IDs range: {int(npz_ids.min())} — {int(npz_ids.max())}')
print(f'Waste range : {npz_waste.min():.2f} — {npz_waste.max():.2f}')
print(f'Max capacity: {npz_cap[0]:.1f}')

### B-3 · Geographic Coordinates

In [ ]:
# Geographic scatter of bins and depot (sample 0)
s0_locs  = npz_locs[0]   # (N, 2): columns are lat, lon
s0_depot = npz_depot[0]  # (2,)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean waste across all days as colour
mean_waste_s0 = npz_waste[0].mean(axis=0)   # (N,)
sc = axes[0].scatter(s0_locs[:,1], s0_locs[:,0],   # lon on x, lat on y
                     c=mean_waste_s0, cmap='YlOrRd', s=70,
                     edgecolors='white', linewidth=0.4, zorder=2,
                     vmin=0, vmax=float(npz_cap[0]))
fig.colorbar(sc, ax=axes[0], label='Mean waste fill level (% capacity)', shrink=0.8)
axes[0].scatter(s0_depot[1], s0_depot[0], c='blue', s=300, marker='*',
               edgecolors='black', linewidth=0.8, zorder=3, label='Depot')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
axes[0].set_title(f'Bin Locations — Sample 0 (colour = mean waste, {N_B} bins)')
axes[0].legend()

# Distribution of distances from depot (geographic units ≈ degrees)
dist_to_depot = np.linalg.norm(npz_locs - npz_depot[:,None,:], axis=-1)  # (S, N)
axes[1].hist(dist_to_depot.ravel(), bins=40, color='steelblue', alpha=0.8, edgecolor='white')
axes[1].axvline(dist_to_depot.mean(), color='red', linestyle='--',
                label=f'Mean={dist_to_depot.mean():.4f}°')
axes[1].set_title('Distance to Depot (geographic degrees)')
axes[1].set_xlabel('Distance (°)'); axes[1].set_ylabel('Count'); axes[1].legend()

plt.tight_layout(); plt.show()

### B-4 · Waste Fill-Level Time-Series

In [ ]:
days = np.arange(T_D)

# ── Aggregate across bins: mean/std fill per day ───────────────────────────────
mean_day = npz_waste[0].mean(axis=1)   # (T,) – mean across bins per day
std_day  = npz_waste[0].std(axis=1)    # (T,) – std across bins per day

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(days, mean_day, color='steelblue', lw=2, label='Mean fill')
axes[0].fill_between(days, mean_day-std_day, mean_day+std_day, alpha=0.25,
                     color='steelblue', label='±1 std')
axes[0].set_title('Average Bin Fill Level Over Days (sample 0)')
axes[0].set_xlabel('Day'); axes[0].set_ylabel('Fill level (% capacity)')
axes[0].legend()

# ── Per-bin trajectories (random sample of bins) ───────────────────────────────
N_SHOW = min(20, N_B)
bin_picks = np.random.default_rng(SEED).choice(N_B, N_SHOW, replace=False)
cm_bins = plt.cm.tab20(np.linspace(0, 1, N_SHOW))
for bi, col in zip(bin_picks, cm_bins):
    axes[1].plot(days, npz_waste[0, :, bi], color=col, lw=1, alpha=0.75,
                 label=f'Bin {bi}')
axes[1].set_title(f'Per-Bin Fill Trajectories — {N_SHOW} random bins (sample 0)')
axes[1].set_xlabel('Day'); axes[1].set_ylabel('Fill level (% capacity)')

plt.tight_layout(); plt.show()

In [ ]:
# ── Heatmap: bins × days ─────────────────────────────────────────────────────
MAX_BINS_HEAT = 60
step_h = max(1, N_B // MAX_BINS_HEAT)
bin_sel = np.arange(0, N_B, step_h)
heat = npz_waste[0, :, bin_sel].T   # (bins_shown, T)

fig, ax = plt.subplots(figsize=(max(12, T_D * 0.5), max(5, len(bin_sel) * 0.18)))
im = ax.imshow(heat, aspect='auto', cmap='YlOrRd', vmin=0, vmax=float(npz_cap[0]))
fig.colorbar(im, ax=ax, label='Fill level (% capacity)', shrink=0.8)
ax.set_yticks(range(len(bin_sel)))
ax.set_yticklabels(bin_sel, fontsize=7)
ax.set_xlabel('Day'); ax.set_ylabel('Bin index (sampled)')
ax.set_title(f'Waste Fill Heatmap — Bins × Days (sample 0, {N_B} bins, {T_D} days)')
plt.tight_layout(); plt.show()

# Per-bin fill statistics across all days
bin_means = npz_waste[0].mean(axis=0)   # (N,)
bin_stds  = npz_waste[0].std(axis=0)
bin_peaks = npz_waste[0].max(axis=0)

stat_df = pd.DataFrame({
    'Mean fill': np.round(bin_means, 2),
    'Std fill':  np.round(bin_stds, 2),
    'Peak fill': np.round(bin_peaks, 2),
    'Min fill':  np.round(npz_waste[0].min(axis=0), 2),
    'Node ID':   npz_ids[0],
}, index=pd.Index(range(N_B), name='Bin'))
print(f'Per-bin fill statistics (first 10 / {N_B} bins shown):')
display(stat_df.head(10))

### B-5 · Noisy vs Clean Waste

In [ ]:
# ── Noise magnitude: |noisy - clean| ─────────────────────────────────────────
noise_abs = np.abs(npz_noisy - npz_waste)   # (S, T, N)
noise_rel = noise_abs / (npz_waste + 1e-6)  # relative to clean fill level

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Distribution of absolute noise
axes[0].hist(noise_abs.ravel(), bins=60, color='tomato', alpha=0.8, edgecolor='white')
axes[0].set_title('Absolute Noise  |noisy − clean|')
axes[0].set_xlabel('Fill level difference (% capacity)'); axes[0].set_ylabel('Count')

# Noise over days (mean across bins)
noise_per_day = noise_abs[0].mean(axis=1)   # (T,)
axes[1].plot(days, noise_per_day, color='tomato', lw=2)
axes[1].fill_between(days, 0, noise_per_day, alpha=0.2, color='tomato')
axes[1].set_title('Mean Absolute Noise per Day (sample 0)')
axes[1].set_xlabel('Day'); axes[1].set_ylabel('Mean |noise| (% capacity)')

# Clean vs noisy trajectories for 3 example bins
sample_bins = np.random.default_rng(SEED+1).choice(N_B, 3, replace=False)
for bi, col in zip(sample_bins, ['steelblue', 'darkorange', 'mediumseagreen']):
    axes[2].plot(days, npz_waste[0, :, bi], color=col, lw=2, label=f'Clean bin {bi}')
    axes[2].plot(days, npz_noisy[0, :, bi], color=col, lw=1.5, ls='--', alpha=0.7,
                 label=f'Noisy bin {bi}')
axes[2].set_title('Clean vs Noisy Fill (3 sample bins)')
axes[2].set_xlabel('Day'); axes[2].set_ylabel('Fill level (% capacity)')
axes[2].legend(fontsize=7, ncol=2)

plt.suptitle(f'Sensor Noise Analysis — {os.path.basename(NPZ_PATH)}', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

print(f'Noise stats: mean={noise_abs.mean():.3f}  max={noise_abs.max():.3f}  '
      f'rel_mean={noise_rel.mean():.3f}')

### B-6 · Multi-Dataset Comparison (NPZ)

In [ ]:
# ── Compare across bin counts and/or distributions ────────────────────────────
NPZ_CONFIGS = [
    (NPZ_AREA, 100,  NPZ_DISTRIBUTION, NPZ_DAYS),
    (NPZ_AREA, 170,  NPZ_DISTRIBUTION, NPZ_DAYS),
    (NPZ_AREA, 100,  'emp',            NPZ_DAYS),
    (NPZ_AREA, 170,  'emp',            NPZ_DAYS),
]

npz_datasets = {}
for (area, nb, dist, nd) in NPZ_CONFIGS:
    p = os.path.join(PROJECT_ROOT, 'data', 'wsr_simulator', 'datasets',
                     f'{area}{nb}_{dist}_wsr{nd}_N{NPZ_N_SAMPLES}_seed{NPZ_SEED}.npz')
    if os.path.exists(p):
        ds = np.load(p, allow_pickle=True)
        label = f'{nb}bins-{dist}'
        npz_datasets[label] = ds
        print(f'  {label}: waste {ds["waste"].shape}')
    else:
        print(f'  {nb}bins-{dist}: not found')

if len(npz_datasets) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    for (lbl, ds), col in zip(npz_datasets.items(), COLORS):
        w    = ds['waste']        # (S, T, N)
        flat = w.ravel()
        x_r  = np.linspace(float(flat.min()), float(flat.max()), 300)
        kde  = gaussian_kde(np.random.choice(flat, min(500000, len(flat)),
                                             replace=False), bw_method='scott')(x_r)
        axes[0].plot(x_r, kde, color=col, lw=2, label=lbl)

        day_means = w[0].mean(axis=1)   # (T,) mean across bins per day
        axes[1].plot(np.arange(w.shape[1]), day_means, color=col, lw=2, label=lbl)

    axes[0].set_title('Waste Fill KDE — Multi-Dataset Comparison')
    axes[0].set_xlabel('Fill level (% capacity)'); axes[0].set_ylabel('Density')
    axes[0].legend(fontsize=9)

    axes[1].set_title('Daily Mean Fill Level — Multi-Dataset')
    axes[1].set_xlabel('Day'); axes[1].set_ylabel('Mean fill (% capacity)')
    axes[1].legend(fontsize=9)

    plt.tight_layout(); plt.show()